# Exploring the extracted entities/relations with pandas

This loads the same data that's in Neo4j (`data/extraction_all.jsonl`) directly into
DataFrames, for statistical/distributional questions that are easier to answer as a
table than by navigating the graph browser -- e.g. "what are the most common entities",
"how are relation types distributed". For structural questions ("what connects to what",
multi-hop traversal), the graph in Neo4j is still the right tool -- this notebook is a
complement to that, not a replacement.

In [1]:
import json
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd().parent
EXTRACTION_PATH = PROJECT_ROOT / "data" / "extraction_all.jsonl"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

## Load into two flat DataFrames: one row per entity mention, one row per relation

In [2]:
records = [json.loads(line) for line in EXTRACTION_PATH.read_text(encoding="utf-8").splitlines() if line.strip()]

entity_rows = []
relation_rows = []
for r in records:
    chunk_id = r["chunk_id"]
    celex = chunk_id.split("_recital_")[0].split("_article_")[0]
    for e in r["entities"]:
        entity_rows.append({"chunk_id": chunk_id, "celex": celex, "text": e["text"], "type": e["type"]})
    for rel in r["relations"]:
        relation_rows.append({"chunk_id": chunk_id, "celex": celex, **rel})

entities_df = pd.DataFrame(entity_rows)
relations_df = pd.DataFrame(relation_rows)

# normalized column mirrors the dedup key used when loading into Neo4j
entities_df["normalized"] = entities_df["text"].str.strip().str.lower()

print(f"{len(records)} chunks, {len(entities_df)} entity mentions, {len(relations_df)} relations")
entities_df.head()

1050 chunks, 2397 entity mentions, 434 relations


,chunk_id,celex,text,type,normalized
0,32016R0679_recital_1_0,32016R0679,fundamental right,LEGAL_CONCEPT,fundamental right
1,32016R0679_recital_1_0,32016R0679,'Charter',LEGAL_ACT,'charter'
2,32016R0679_recital_1_0,32016R0679,'Treaty on the Functioning of the European Uni...,LEGAL_ACT,'treaty on the functioning of the european uni...
3,32016R0679_recital_2_0,32016R0679,personal data,LEGAL_CONCEPT,personal data
4,32016R0679_recital_3_0,32016R0679,Directive 95/46/EC,LEGAL_ACT,directive 95/46/ec


## Document titles, for readable labels instead of bare CELEX codes

In [3]:
titles = {}
for f in PROCESSED_DIR.glob("*.json"):
    doc = json.loads(f.read_text(encoding="utf-8"))
    titles[doc["celex"]] = doc["title"][:60]

entities_df["document"] = entities_df["celex"].map(titles)
relations_df["document"] = relations_df["celex"].map(titles)
titles

{'32022L2555': 'DIRECTIVE (EU) 2022/2555 OF THE EUROPEAN PARLIAMENT AND OF T',
 '32019R0881': 'REGULATION (EU) 2019/881 OF THE EUROPEAN PARLIAMENT AND OF T',
 '32016R0679': 'REGULATION (EU) 2016/679 OF THE EUROPEAN PARLIAMENT AND OF T',
 '32022R0868': 'REGULATION (EU) 2022/868 OF THE EUROPEAN PARLIAMENT AND OF T'}

## Entity counts by type

In [4]:
entities_df["type"].value_counts()

type
ORGANIZATION     689
LEGAL_ACT        626
LEGAL_CONCEPT    605
OBLIGATION       310
PENALTY          137
DATE              30
Name: count, dtype: int64

## Most frequently mentioned entities overall (deduped by normalized text)

In [5]:
entities_df.groupby(["normalized", "type"]).size().sort_values(ascending=False).head(20)

normalized                  type         
enisa                       ORGANIZATION     79
member states               ORGANIZATION     69
supervisory authority       ORGANIZATION     64
personal data               LEGAL_CONCEPT    62
commission                  ORGANIZATION     56
controller                  ORGANIZATION     27
directive                   LEGAL_ACT        26
regulation (eu) 2016/679    LEGAL_ACT        20
member state                ORGANIZATION     17
regulation                  LEGAL_ACT        16
board                       ORGANIZATION     15
data subject                LEGAL_CONCEPT    14
commission                  LEGAL_ACT        14
competent authorities       ORGANIZATION     13
public sector bodies        ORGANIZATION     13
member states               LEGAL_CONCEPT    13
management board            ORGANIZATION     12
supervisory authority       LEGAL_ACT        12
supervisory authorities     ORGANIZATION     11
lead supervisory authority  ORGANIZATION     1

## Relation type distribution

In [6]:
relations_df["relation"].value_counts()

relation
APPLIES_TO               185
IMPOSES_OBLIGATION_ON     65
RESPONSIBLE_FOR           64
REFERENCES                64
ESTABLISHES               21
GRANTS_RIGHT_TO           12
DEFINES                   10
REPEALS                   10
SUBJECT_TO                 3
Name: count, dtype: int64

## Which entities appear in more than one document?

The cross-document overlap that's hard to notice reading four separate PDFs.

In [7]:
doc_counts = entities_df.groupby("normalized")["document"].nunique()
shared = doc_counts[doc_counts > 1].sort_values(ascending=False)

shared_detail = (
    entities_df[entities_df["normalized"].isin(shared.index)]
    .groupby("normalized")["document"]
    .unique()
)
shared_detail.loc[shared.index]

normalized
internal market                           [REGULATION (EU) 2016/679 OF THE EUROPEAN PARL...
european parliament and of the council    [REGULATION (EU) 2016/679 OF THE EUROPEAN PARL...
regulation (eu) 2016/679                  [REGULATION (EU) 2016/679 OF THE EUROPEAN PARL...
regulation (eu)                           [REGULATION (EU) 2016/679 OF THE EUROPEAN PARL...
commission                                [REGULATION (EU) 2016/679 OF THE EUROPEAN PARL...
                                                                ...                        
delegated acts                            [REGULATION (EU) 2016/679 OF THE EUROPEAN PARL...
data subjects                             [REGULATION (EU) 2016/679 OF THE EUROPEAN PARL...
data subject                              [REGULATION (EU) 2016/679 OF THE EUROPEAN PARL...
data protection rules                     [REGULATION (EU) 2016/679 OF THE EUROPEAN PARL...
vulnerabilities                           [REGULATION (EU) 2019/881 O

## Entity/relation volume per document

Which document contributed the most extractable structure -- not necessarily the longest
document, since that depends on how entity-dense its language is.

In [8]:
pd.DataFrame({
    "entities": entities_df.groupby("document").size(),
    "relations": relations_df.groupby("document").size(),
})

,entities,relations
document,,
DIRECTIVE (EU) 2022/2555 OF THE EUROPEAN PARLIAMENT AND OF T,673,107
REGULATION (EU) 2016/679 OF THE EUROPEAN PARLIAMENT AND OF T,791,152
REGULATION (EU) 2019/881 OF THE EUROPEAN PARLIAMENT AND OF T,534,100
REGULATION (EU) 2022/868 OF THE EUROPEAN PARLIAMENT AND OF T,399,75
